# Kandinsky V2A — Audio Quality and Context Prompt Pilot

## Question
Can a more focused prompt improve sound quality and scene fit for the **same video**, without changing the model or generation settings?

## Controlled comparison
- One user-selected, silent five-second video window
- Kandinsky Full Lite, loaded once
- The same random seed for all three candidates
- Only the prompt framing changes

This is a **prompt-framing experiment inside Kandinsky V2A**, not a comparison between different models. It does not train or fine-tune anything. Audio/video synchronization is recorded as a guardrail; the main listening criteria are event match, naturalness, noise/artifacts, and unwanted music or speech.

The checkpoint needs an 80 GB-class GPU for this setup. The notebook stops early if the runtime has less than 76,000 MiB of GPU memory or 45 GiB of free disk.

In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from IPython.display import Audio, FileLink, Markdown, display

if sys.version_info < (3, 13):
    raise RuntimeError(f"Use a Python 3.13+ Colab runtime. Found: {sys.version}")

def run_checked(command, cwd=None, label="command", env=None):
    result = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-2500:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"{label} failed.")
    return result

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU runtime. Select a Colab GPU and reconnect.")

gpu_result = run_checked([
    "nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
    "--format=csv,noheader,nounits"
], label="inspect GPU")
gpu_fields = next(csv.reader([gpu_result.stdout.strip()]))
GPU_NAME = gpu_fields[0].strip()
GPU_MEMORY_MIB = int(gpu_fields[1])
GPU_COMPUTE = gpu_fields[2].strip() if len(gpu_fields) > 2 else "unknown"
print(f"GPU: {GPU_NAME} | {GPU_MEMORY_MIB:,} MiB | compute {GPU_COMPUTE}")
if GPU_MEMORY_MIB < 76000:
    raise RuntimeError("Full Lite needs an 80 GB-class GPU for this experiment. Reconnect with an eligible Colab GPU.")

# Set VIDEO_PATH to the selected file's exact Colab path if more than one video is listed.
VIDEO_PATH = None
# Describe only the main audible event that is actually visible in this clip, in one short English sentence.
EVENT_DESCRIPTION = ""
WINDOW_START_S = 0.0
DURATION_S = 5  # Kandinsky time_length must be an integer
SEED = 42

VIDEO_SUFFIXES = {".mp4", ".mov", ".webm", ".mkv"}
video_candidates = sorted(
    p for p in Path("/content").iterdir()
    if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES
    and not p.name.lower().startswith(("kandinsky_v2a_", "candidate_", "output_"))
)
if VIDEO_PATH is None:
    if len(video_candidates) == 1:
        VIDEO_PATH = video_candidates[0]
    else:
        print("Video files found in /content:")
        for path in video_candidates:
            print(" -", path)
        raise RuntimeError("Set VIDEO_PATH to the selected video's exact path, then rerun this cell.")

VIDEO_PATH = Path(VIDEO_PATH)
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
if not EVENT_DESCRIPTION.strip():
    raise ValueError("Fill EVENT_DESCRIPTION with one short, factual English description of the visible event.")

scratch = Path("/content/local-scratch")
storage_options = [p for p in (Path("/content"), scratch) if p.exists()]
CACHE_ROOT = max(storage_options, key=lambda p: shutil.disk_usage(p).free)
FREE_GIB = shutil.disk_usage(CACHE_ROOT).free / (1024**3)
if FREE_GIB < 45:
    raise RuntimeError(f"At least 45 GiB free disk is required. Found {FREE_GIB:.1f} GiB on {CACHE_ROOT}.")

WORK = CACHE_ROOT / "kandinsky_audio_quality_context_pilot"
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "outputs"
for folder in (INPUT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
RESULTS_ZIP = Path("/content/kandinsky_audio_quality_context_results.zip")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_probe = run_checked([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(VIDEO_PATH)
], label="inspect selected video")
SOURCE_DURATION_S = float(source_probe.stdout.strip())
if WINDOW_START_S < 0 or WINDOW_START_S + DURATION_S > SOURCE_DURATION_S:
    raise ValueError(f"Selected window {WINDOW_START_S:.2f}–{WINDOW_START_S + DURATION_S:.2f}s exceeds the {SOURCE_DURATION_S:.2f}s video.")

SILENT_VIDEO = INPUT_DIR / "selected_window_5s_silent.mp4"
run_checked([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", str(WINDOW_START_S), "-i", str(VIDEO_PATH), "-t", str(DURATION_S),
    "-map", "0:v:0", "-an",
    "-vf", "fps=24,scale='min(960,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
    "-pix_fmt", "yuv420p", str(SILENT_VIDEO)
], label="prepare silent five-second video")
window_probe = run_checked([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(SILENT_VIDEO)
], label="verify prepared window")
WINDOW_DURATION_S = float(window_probe.stdout.strip())
if abs(WINDOW_DURATION_S - DURATION_S) > 0.05:
    raise RuntimeError(f"Prepared window is {WINDOW_DURATION_S:.3f}s; expected five seconds.")

VIDEO_SHA256 = sha256_file(SILENT_VIDEO)
print(f"Source: {VIDEO_PATH.name} | {SOURCE_DURATION_S:.2f}s")
print(f"Window: {WINDOW_START_S:.2f}s + {WINDOW_DURATION_S:.2f}s | SHA-256: {VIDEO_SHA256}")
print(f"Work folder: {WORK} | free disk: {FREE_GIB:.1f} GiB")

## Step 1 — Prepare the exact input

If the cell lists multiple videos, set `VIDEO_PATH` to the chosen file path shown in the output and rerun the cell. Enter a short English sentence for `EVENT_DESCRIPTION`; describe what is visible, not what you hope the model will invent. Set `WINDOW_START_S` if the useful event is not in the first five seconds.

The source audio, if any, is removed. All three generations receive identical silent video frames.

## Step 2 — Install the pinned Kandinsky environment

This uses the same tested Kandinsky source revision and Full Lite configuration as the persistent-worker pilot. The pipeline will be loaded once and reused for all three generations.

In [ ]:
SOURCE_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
REPO = Path("/content/kandinsky-v2a-worker-src")
KANDINSKY_HOME = CACHE_ROOT / "kandinsky-model-cache"
os.environ["KANDINSKY_HOME"] = str(KANDINSKY_HOME)

if not (REPO / ".git").exists():
    run_checked([
        "git", "clone", "--filter=blob:none",
        "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)
    ], label="clone pinned Kandinsky source")

run_checked(["git", "fetch", "origin", SOURCE_COMMIT], cwd=REPO, label="fetch pinned source")
run_checked(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO, label="checkout pinned source")

if not shutil.which("uv"):
    run_checked([sys.executable, "-m", "pip", "install", "-q", "uv"], label="install uv")
if not shutil.which("just"):
    run_checked([
        "bash", "-lc",
        "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"
    ], label="install just")

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
if setup.returncode:
    print(setup.stdout[-2500:])
    print(setup.stderr[-5000:])
    raise RuntimeError("Kandinsky dependency setup failed.")

PY = REPO / ".venv" / "bin" / "python"
if not PY.is_file():
    raise FileNotFoundError(f"Pinned environment was not created: {PY}")

CONFIG = REPO / "audio_quality_context_lite.yaml"
CONFIG.write_text("""checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""", encoding="utf-8")

run_checked(["just", "download", "lite"], cwd=REPO, label="download Full Lite checkpoint")
print("Pinned Kandinsky source:", SOURCE_COMMIT)
print("Worker Python:", PY)
print("Configuration:", CONFIG)
print("Checkpoint: Kandinsky 6.0 Lite (Full Lite)")

## Step 3 — Generate three prompt variants

The only experimental variable is prompt framing:

- **A — Video-led control:** asks for natural sound matching the visible action.
- **B — Foreground event:** emphasizes a clear, natural foreground event.
- **C — Scene context:** keeps the event prominent and permits only subtle fitting ambience.

All prompts use the event description you entered, the same video, the same Full Lite checkpoint, and seed 42. They run sequentially in one process; they are not parallelized.

In [ ]:
PROMPTS = [
    {
        "id": "A_video_led",
        "label": "A — Video-led control",
        "text": f"Generate natural sound that matches the visible events in this video. {EVENT_DESCRIPTION} No music. No speech."
    },
    {
        "id": "B_foreground_event",
        "label": "B — Foreground event",
        "text": f"A clear, natural, close sound of this visible event: {EVENT_DESCRIPTION} Keep the main event prominent and realistic. No music. No speech."
    },
    {
        "id": "C_scene_context",
        "label": "C — Event with scene context",
        "text": f"A natural soundscape for this exact video. The main visible event is: {EVENT_DESCRIPTION} Keep that event clear in the foreground, with only subtle background ambience that fits the scene. No music. No speech."
    }
]
for item in PROMPTS:
    print(f"{item['label']}\n{item['text']}\n")

DRIVER = WORK / "generate_candidates.py"
SPEC_PATH = WORK / "generation_spec.json"
REPORT_PATH = WORK / "generation_report.json"

driver_source = r'''import hashlib, json, sys, time, traceback
from pathlib import Path
import torch
from kandinsky import get_pipeline

spec_path = Path(sys.argv[1])
spec = json.loads(spec_path.read_text(encoding="utf-8"))
output_dir = Path(spec["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
report_path = Path(spec["report_path"])

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

report = {
    "schema": "kandinsky_v2a_audio_quality_context_pilot_v1",
    "source_commit": spec["source_commit"],
    "checkpoint": "Kandinsky 6.0 Lite (Full Lite)",
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "input_video": Path(spec["video"]).name,
    "input_video_sha256": spec["video_sha256"],
    "duration_seconds": spec["duration_seconds"],
    "seed": spec["seed"],
    "prompts": spec["prompts"],
    "pipeline_load_seconds": None,
    "runs": [],
    "error": None
}

def save_report():
    temp = report_path.with_suffix(".tmp")
    temp.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(report_path)

try:
    torch.cuda.synchronize()
    started = time.perf_counter()
    pipe = get_pipeline(spec["config"], device="cuda", attention_engine="sdpa")
    torch.cuda.synchronize()
    report["pipeline_load_seconds"] = round(time.perf_counter() - started, 2)
    print(f"Pipeline loaded in {report['pipeline_load_seconds']}s on {report['gpu']}", flush=True)
    save_report()

    for index, item in enumerate(spec["prompts"], start=1):
        output_path = output_dir / f"candidate_{index}_{item['id']}.mp4"
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        pipe(
            text=item["text"],
            video=spec["video"],
            time_length=spec["duration_seconds"],
            seed=spec["seed"],
            save_path=str(output_path),
            show_progress=True,
        )
        torch.cuda.synchronize()
        elapsed = round(time.perf_counter() - started, 2)
        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(f"Expected output was not written: {output_path}")
        report["runs"].append({
            "id": item["id"],
            "label": item["label"],
            "seconds": elapsed,
            "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / (1024**2), 1),
            "peak_reserved_mib": round(torch.cuda.max_memory_reserved() / (1024**2), 1),
            "output_mp4": output_path.name,
            "output_sha256": sha256_file(output_path),
            "output_bytes": output_path.stat().st_size
        })
        save_report()
        print(f"Finished {item['label']}: {elapsed}s", flush=True)
except Exception as exc:
    report["error"] = f"{type(exc).__name__}: {exc}"
    report["traceback"] = traceback.format_exc()
    save_report()
    raise
'''

DRIVER.write_text(driver_source, encoding="utf-8")
SPEC = {
    "source_commit": SOURCE_COMMIT,
    "video": str(SILENT_VIDEO),
    "video_sha256": VIDEO_SHA256,
    "duration_seconds": DURATION_S,
    "seed": SEED,
    "prompts": PROMPTS,
    "config": str(CONFIG),
    "output_dir": str(OUTPUT_DIR),
    "report_path": str(REPORT_PATH)
}
SPEC_PATH.write_text(json.dumps(SPEC, ensure_ascii=False, indent=2), encoding="utf-8")

run = subprocess.run([str(PY), str(DRIVER), str(SPEC_PATH)], cwd=REPO, env=os.environ.copy())
if run.returncode:
    raise RuntimeError(f"Generation worker exited with code {run.returncode}. Inspect {REPORT_PATH} and rerun logs above.")
if not REPORT_PATH.is_file():
    raise FileNotFoundError(f"Generation report was not created: {REPORT_PATH}")
print("All three candidates are ready:", OUTPUT_DIR)

## Step 4 — Extract and listen to audio only

The MP4 files preserve the generated result for inspection. This cell extracts their audio tracks as WAVs so you can compare sound without the video playing. Rate each candidate before deciding which prompt framing sounds best.

In [ ]:
REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
AUDIO_DIR = OUTPUT_DIR / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

for run_info in REPORT["runs"]:
    mp4_path = OUTPUT_DIR / run_info["output_mp4"]
    wav_path = AUDIO_DIR / (Path(run_info["output_mp4"]).stem + ".wav")
    extraction = subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(mp4_path), "-map", "0:a:0", "-vn",
        "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(wav_path)
    ], capture_output=True, text=True)
    if extraction.returncode:
        print(extraction.stderr[-2000:])
        raise RuntimeError(f"Could not extract audio from {mp4_path.name}")
    run_info["output_wav"] = wav_path.name
    run_info["wav_sha256"] = sha256_file(wav_path)
    display(Markdown(f"### {run_info['label']} — {run_info['seconds']}s"))
    display(Audio(filename=str(wav_path)))

REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
print("Audio-only listening is ready.")

## Step 5 — Record your ratings and package the results

Use a 1–5 scale for event match, naturalness, and artifacts. For artifacts, **1 means clean** and **5 means severe**. Also note unwanted music/speech and whether you would keep the result.

The package includes the generated MP4/WAV files and reports, but **does not include the source video**. The source video remains only in this Colab runtime.

In [ ]:
REVIEW_PATH = WORK / "listening_ratings.json"
review = {
    "schema": "kandinsky_v2a_prompt_comparison_ratings_v1",
    "input_video_sha256": VIDEO_SHA256,
    "same_seed_for_all": SEED,
    "scale_note": "event_match, naturalness: 1=poor, 5=strong; artifacts: 1=clean, 5=severe",
    "ratings": {
        run_info["id"]: {
            "event_match_1_to_5": None,
            "naturalness_1_to_5": None,
            "artifacts_1_to_5": None,
            "unwanted_music_or_speech": None,
            "would_keep": None,
            "notes": ""
        }
        for run_info in REPORT["runs"]
    }
}
REVIEW_PATH.write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding="utf-8")

with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, arcname="generation_report.json")
    archive.write(REVIEW_PATH, arcname="listening_ratings_template.json")
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=str(Path("outputs") / path.relative_to(OUTPUT_DIR)))

print("Results ZIP:", RESULTS_ZIP)
print("ZIP size: %.1f MiB" % (RESULTS_ZIP.stat().st_size / (1024**2)))
display(FileLink(str(RESULTS_ZIP)))

## How to interpret the result

This pilot answers a narrow question: **does prompt framing change audio quality or scene fit for this one clip?** It cannot establish that one prompt works for every scene, and it does not compare different model architectures.

If B is clearer but C is more scene-appropriate, the next product step is to let the user choose between a clean foreground event and a fuller contextual sound. If all three remain noisy, prompt wording is probably not enough; we should compare a different audio-generation path or a checkpoint/inference configuration in a separate controlled test.

No training is performed. The notebook uses the Kandinsky source pinned to commit `7ceeefad7fb625011f7db602c43893a989f19c15` and references the [official video-to-audio mode documentation](https://github.com/kandinskylab/kandinsky-6/blob/main/docs/generation_modes.md).